<a href="https://colab.research.google.com/github/osciss/project_hardwaresec/blob/main/fi_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to the IL1333 project! Read through the instructions carefully.
In this project you will get familiarity with how hardware attacks, in this case fault injection, can be used to break secure software implementations.

## How to use this notebook
In order to run this lab in Colab, you will need to sign in with a Google account. If you do not already have a Google account, you can [create one using your KTH email](https://accounts.google.com/SignUpWithoutGmail). **Your changes to this notebook will not be saved unless you create a copy!** Create a copy using "File > Save a copy in Drive" and use only this copy going forward.

If you do not want to create a Google account, you can download the Jupyter notebook using "File > Download > Download .ipynb" and run it locally in whichever way you prefer. Note that we are not able to offer support if you run into issues with such a setup.

## Rules for this lab

### Challenge rules
This project requires some understanding of the ARM assembly language. You are allowed to consult an LLM for assistance in understanding the assembly. If you have no previous experience with assembly language, we encourage you to take advantage of an LLM, as modern ones are generally good at explaining short assembly snippets, so long as you verify what it is telling you.

How you find your solution is not important, only that you understand it.

### Report rules
You are **NOT** allowed to write the report using an LLM. You are allowed to consult one to help with your understanding, **but the text must be your own words**.

## Project report
You must submit a well-structured report, written in your own words (**again, no AI generated text**), that answers the following questions:
- For at least *five of the six* challenges:
  - What was the **vulnerability** you identified?
  - How did you **exploit it**?
  - What was the instruction **offset**?
  - How did fault injection **make this attack possible**?
- Come up with at least one challenge **idea of your own**. Could you imagine seeing this in a real embedded device?
- How **realistic** do you think these kind of attacks would be in a real setting?
- What **constrains** you as an attacker? Feel free to google.
- How can you **protect** against fault injection? (software and hardware)
- How can we make this project more pedagogical and more enjoyable for next year? We really appreciate your feedback. :)

Regarding word-count: You will not be graded on the length of your report but on the content. If you must have an answer, aim at about two pages, not counting figures and code listings.

## Questions
If you have questions regarding the project, please use the **discussion forum** or email Linus (TA) at *lbackl@kth.se*.

# Setup

First, install the required libraries by running the cell below.

In [ ]:
!echo -e "\033[1;33mDownloading package...\033[0m" && wget -O filab-tools.zip "https://kth-my.sharepoint.com/:u:/g/personal/dubrova_ug_kth_se/IQAoJXSts4BKTZ1KjVdkKIwdAdPSEyiNe8bgGRNM-APYXVc?download=1" \
&& echo -e "\033[1;33mInstalling libraries...\033[0m" && pip install ./filab-tools.zip \
&& echo -e "\033[1;32mSuccessfully finished setup!\033[0m"

from filab import *

If the output says "Successfully finished setup!", everything worked correctly and you can proceed with the project. Otherwise, please use the discussion forum to get help.

# Overview
Since we don't have enough hardware kits and due to the unreliable nature of fault injection, we will use an emulation framework for the IL1333 fault injection project.

# Scenario
Your target is an emulated **32-bit ARM microcontroller** similar to an STM32. Your fault injection tool has been perfectly calibrated to precisely **skip one instruction** during the execution of its firmware. You can therefore time the glitch simply by counting instructions. Imagine the microcontroller runs some kind of access system and controls the electronic doors to a bank-vault or something cooler.

# The challenges
**There are six challenges of which you must solve and include at least five in your report.** Each challenge emulates a situation where you use fault injection to bypass the access control system. There are several solutions to some of the challenges. **You only need to find one solution to each challenge**, though you are encouraged to look for more. **Make sure you properly understand your solution that you present in your report.**

# Challenge 0: Tutorial

_This challenge is designed for you to get used to both the format of the challenges and to the basics of instruction skipping using fault injection. For each challenge, we will provide you with both a short description of the code that you are going to attack and a goal for your attack, to get you moving in the right direction._

</br>

This first challenge is the most basic example of instruction skipping fault injection. The function initialises a value to 0, increments it 10 times, and returns 0 if the value afterwards is 10 and 1 if it is a different value. Under regular program execution, this function can therefore never return 1.

**Goal**: Can you use fault injection to force it to do so anyway?

</br>

_After the introduction, there will be a block with some code. For each challenge, you are given:_

- _**Assembly code** that is your target function_
- _**C code** to help you understand the assembly_
- _The **initial state** where the emulation of the function starts_
- _The **success state** that you must reach when the function exists to complete the challenge._

_Since this code defines the actual challenge, changing any of it can make the challenge much easier or more difficult (or even impossible) to solve. You are therefore **not allowed to change the challenge definitions!**_

In [ ]:
# Challenge 0
# Run this block to pretty-print!

c_code = '''\
int func() {
    int value = 0;
    value += 1;
    value += 1;
    value += 1;
    value += 1;
    value += 1;
    value += 1;
    value += 1;
    value += 1;
    value += 1;
    value += 1;
    return value != 10;
}
'''

asm_code = '''\
mov r1, #0      // Initialise r1 to 0
add r1, #1      // Increment r1 by one
add r1, #1
add r1, #1
add r1, #1
add r1, #1
add r1, #1
add r1, #1
add r1, #1
add r1, #1
add r1, #1
cmp r1, #10     // Compare r1 and 10
ite ne          // If previous comparison was equal, then
movne r0, #1    // load value 1 into Return Register (r0)
moveq r0, #0    // otherwise load value 0 into Return Register (r0)
bx lr           // Return
'''

initial_state = {
    'registers': {
        'r0': 0x0,
        'r1': 0x0,
        'lr': RETURN_ADDRESS
    }
}

success_state = {
    'registers': {
        'r0': 0x1
    }
}

challenge_0 = {
    'index': 0,
    'c_code': c_code,
    'asm_code': asm_code,
    'initial_state': initial_state,
    'success_state': success_state
}

_Since this code can be difficult to read, we have implemented a simple form of pretty-printing that provides some syntax highlighting. Run the cell below and look at the output._

In [ ]:
print_challenge(challenge_0)

_Brief comment: If you look closely, you may notice that the assembly instructions in the code block above look different from those in the printed output. The reason for this is that the printing uses the disassembled form of the assembly code you saw earlier, thus the actual instructions depend on how they are assembled. For example, instead of the `mov r1, #0` instruction, the printing shows `mov.w r1, #0`. These instructions are functionally equivalent - If you are unfamiliar with ARM assembly, we strongly recommend that you consult an LLM for help. The numbers to the left of the assembly instructions show the addresses of the instructions in memory, which can be useful for understanding where branches jump to._

_Before we get to the fun part of this project, the actual fault injection, we will briefly explain how the emulation process works._

_Put simply, our framework first initialises the emulator according to the defined **initial state**. That means that the emulator will ensure that registers and memory locations contain the correct values for you to be able to solve the challenge. It then assembles and emulates the execution of the **assembly code**. Once the function returns, the final state is compared against the **success state** and it is decided whether or not the challenge was completed._

_The **first execution is run with no fault injection** so that you can see what the default behaviour of the function is. Let's run the cell below to see what this looks like. You will be prompted for an input, **for now simply enter 'q' and hit 'Enter' to quit the challenge.**_

In [ ]:
run_challenge(challenge_0, debug=False)

_During the emulation, the emulator prints each instruction that is executed. The number in brackets to the left of the instruction indicates the index of the instruction in the execution sequence. You use this index to tell the emulator which instruction you want to skip (so **to skip the first instruction with index '[00]', you simply enter '0' and 'Enter' in the prompt**). After the index, the emulator also prints the address of the instruction in memory, which can be useful, as well as the instruction it executed._

_After the emulation, the emulator will check whether the final state matches the **success state**. You will see either `>>> RESULT: ACCESS DENIED <<<` or `>>> RESULT: ACCESS GRANTED <<<` depending on the result._

_Fault injection is an inherently iterative process, so you may have to try multiple instructions before you successfully manage to gain access. The emulation therefore resets after each unsuccessful attempt, allowing you to pick a new instruction to skip. Since fault injection will change the sequence of instructions being executed, you may lose track of what is going on and which instruction index is correct. At any point, **entering 'r' followed by 'Enter' will rerun the emulation without performing any fault injection**, thus allowing you to reorient yourself and identify the correct index._

_You should now have enough information to successfully perform your first fault injection attack. Run the cell below and enter the index of an instruction to skip. If you don't succeed at first, try a different instruction. Remember to rerun the emulation using 'r' if you lose track of the instruction indices._

In [ ]:
run_challenge(challenge_0, debug=False)

_Finally, sometimes it can be useful to get some more information about the state of the processor at certain steps throughout the execution. For this, we have implemented a small debugging mode that will print the contents of all relevant registers and memory locations at each step of the emulation. You can enable this mode by running the `run_challenge` function with the `debug=True` argument._

_Be aware that this debugging can be quite slow and the output can be difficult to parse, so you may want to use it sparingly._

In [ ]:
run_challenge(challenge_0, debug=True)

_You should now have enough information to be able to complete the rest of the challenges. Remember that the goal is not just for you to identify the correct instruction to skip, but also to understand why skipping this instruction has the observed behaviour._

_Good luck!_

# Challenge 1: Basic Password Check
The function checks if the password (a 32bit word) is correct by comparing it against the hardcoded password 0x1234. Since you don't know the password you enter something arbitrary, like 0x0BAD and try to bypass the check with a glitch. The result is stored in the AUTH_STATE_ADDR memory location.

**Goal: Bypass the password check without knowing the password.**

In [ ]:
# Challenge 1
# Run this block to pretty-print!

c_code = '''\
int check_password(int input, int *auth_status) {
   int secret = 0x1234;
   if (input == secret) {
      *auth_status = 1; // Unlock
      return 1;
   } else {
      *auth_status = 0; // Lock
      return 0;
   }
}
'''

AUTH_STATE_ADDR = RAM_ADDRESS + 0x100

asm_code = f'''\
  movw r2, #0x1234          // Load hardcoded secret
  cmp r0, r2                // Compare input (r0) with secret (r1)
  beq unlock                // Branch to unlock if equal
  movs r0, #0               // Otherwise r2 = 0 (Fail state)
  b store                   // Branch to store
unlock:
  movs r0, #1               // r2 = 1 (Success state)
store:
  str r0, [r1]              // Store success/fail flag to memory
  bx lr                     // Return
'''

initial_state = {
    'registers': {
        'r0': 0x0BAD,
        'r1': AUTH_STATE_ADDR,
        'r2': 0x0,
        'lr': RETURN_ADDRESS
    },
    'memory': {
        AUTH_STATE_ADDR: b'\x00'
    }
}

success_state = {
    'memory': {
        AUTH_STATE_ADDR: b'\x01'
    }
}

challenge_1 = {
    'index': 1,
    'c_code': c_code,
    'asm_code': asm_code,
    'initial_state': initial_state,
    'success_state': success_state
}

print_challenge(challenge_1)

In [ ]:
run_challenge(challenge_1, debug=False)

# Challenge 2: Pointer Pivot
This function calls another function, depending on your privilege level. It calls one function for admins, and another for guests.

**Goal:** Can you glitch the code to call the admin-function instead of the guest-function?

In [ ]:
# Challenge 2
# Run this block to pretty-print!

DISPATCH_TABLE   = 0x20000500
ADMIN_FUNC_ADDR = 0x08002000
GUEST_FUNC_ADDR = 0x08001000

c_code = '''\
void service_dispatcher(int is_guest, void (**table)()) {
    void (*function)() = table[is_guest];
    function();
}
'''

asm_code = f'''\
push {{lr}}
lsl r0, #2              // Convert privilege level to a byte-offset (r0 << 2)
add r1, r0              // Select which function to load based on privilege
ldr r3, [r1]            // Load the function pointer from the table
blx r3                  // Branch to the function
pop {{pc}}
'''

initial_state = {
    'registers': {
        'r0': 1,               # User is NOT an admin (0=admin, 1=guest)
        'r1': DISPATCH_TABLE,  # Pointer to a table of function pointers
        'lr': RETURN_ADDRESS
    },
    'memory': {
        DISPATCH_TABLE:     ADMIN_FUNC_ADDR.to_bytes(4, 'little'), # The function pointers are located
        DISPATCH_TABLE + 4: GUEST_FUNC_ADDR.to_bytes(4, 'little')  # in the table (a list of pointers)
    }
}

success_state = {
    'registers': {
        'pc': ADMIN_FUNC_ADDR  # Successfully executed Admin function
    }
}

challenge_2 = {
    'index': 2,
    'c_code': c_code,
    'asm_code': asm_code,
    'initial_state': initial_state,
    'success_state': success_state
}

print_challenge(challenge_2)

In [ ]:
run_challenge(challenge_2, debug=False)

# Challenge 3: Could You Repeat That Please?

The function parses an 8-bit ID string from the users access card into a signed 8-bit integer to determine the users privileges. Guest users have *positive* IDs and Admin users have *negative* IDs. You are a guest and your access card has the ID '11010110' (LSB to the left, since it is a string).

**Goal:** Can you make the parser misinterpret your ID as an Admin ID?


In [ ]:
# Challenge 3
# Run this block to pretty-print!

c_code = '''\
int is_admin_badge(char* input_bit_string) {
    int8_t id_val = 0;

    for (int i = 0; i < 8; i++) {
        if (input_bit_string[i] != '0') // Compare character to '0'
            id_val += (1 << i); // Add the parsed bit
    }

    return id_val <= 0;
}\n'''

INPUT_STRING_BASE_ADDR = RAM_ADDRESS + 0x10

asm_code = '''\
mov r3, #0            // Initialise the accumulated value to 0
loop:
  ldrb r2, [r0, r1]   // Load the byte at offset r1 from the input (r0) into r2
  cmp r2, #48         // Compare r2 and '0' in ASCII
  beq is_zero         // Jump to 'is_zero' if they are equal
  mov r2, #1          // Otherwise set r2 to 1
  lsl r2, r1          // and shift it to the left by the current loop index (r1)
  add r3, r2          // and add that value to the accumulated value (r3)
  sxtb r3, r3         // and extract the signed 8-bit value after the addition
is_zero:
  add r1, #1          // Increment the loop index
  cmp r1, #8          // Compare and
  blt loop            // repeat the loop if index is < 8
cmp r3, #0            // Final check, compare r3 and 0
ite le                // by checking r3 <= 0
movle r0, #1          // If yes, store 1 in Return Register (r0)
movgt r0, #0          // Otherwise store 0
bx lr                 // Return
'''

initial_state = {
    'registers': {
        'r0': INPUT_STRING_BASE_ADDR,
        'r1': 0x0,
        'r2': 0x0,
        'r3': 0x0,
        'lr': RETURN_ADDRESS
    },
    'memory': {
        INPUT_STRING_BASE_ADDR: b'1101',
        INPUT_STRING_BASE_ADDR + 4: b'0110'
    }
}

success_state = {
    'registers': {
        'r0': 0x1
    }
}

challenge_3 = {
    'index': 3,
    'c_code': c_code,
    'asm_code': asm_code,
    'initial_state': initial_state,
    'success_state': success_state
}

print_challenge(challenge_3)

In [ ]:
run_challenge(challenge_3, debug=False)

# Challenge 4: Uninitialization
The code moves the hardcoded password from a memory location into a buffer on the stack. You feel like there is some vulnerability here and enter an empty password. The empty password (null-bytes) is then compared against the copied password.

**Goal:** Can you find a way to make the comparison report a match? You can assume
that the stack is zero-initialized.

In [ ]:
# Challenge 4
# Run this block to pretty-print!

c_code = '''\
int verify(char *user_input, char *secret_key) {
    char buffer[8];

    // Copy secret key to local stack buffer
    memcpy(buffer, secret_key, 8);

    // Compare first eight bytes of buffer with user input
    for (int i = 0; i < 8; i++) {
        if (buffer[i] != user_input[i]) {
            return 0; // Denied
        }
    }
    return 1; // Granted
}

void memcpy(char *dest, char *src, int len) {
    for (int i=0; i<len; i++) {
        dest[i] = src[i];
    }
}
'''

INPUT_STRING_BASE_ADDR = RAM_ADDRESS + 0x10
PASSWORD_BASE_ADDR = RAM_ADDRESS + 0x20

asm_code = f'''\
push {{lr}}              // Save return address to stack
sub sp, #8             // Allocate 8-byte buffer on the stack
mov r4, r0             // Backup pointer to user input into r7
mov r0, sp             // Destination for memcpy (r0) is the stack buffer
mov r2, #8             // Set copy length (r2) to 8 bytes
bl memcpy              // Call the memcpy function (r0=dest, r1=src, r2=8)

mov r0, sp             // Reset r0 to point to the stack buffer
mov r1, r4             // Set r1 to point to the user input
mov r2, #8             // Set comparison length to 8 bytes
cmp_loop:
  ldrb r3, [r0], #1    // Load byte from stack buffer, increment pointer
  ldrb r4, [r1], #1    // Load byte from user input, increment pointer
  cmp r3, r4           // Compare the two bytes
  bne denied           // If they differ, jump to 'denied'
  sub r2, #1           // Decrement loop counter
  cmp r2, #0           // Check if 8 bytes have been compared
  bne cmp_loop         // If not, repeat loop

granted:
  mov r0, #1           // Success: set return value to 1
  b exit               // Jump to function exit
denied:
  mov r0, #0           // Failure: set return value to 0
exit:
  add sp, #8           // Deallocate stack buffer
  pop {{pc}}             // Restore lr into pc to return to caller

memcpy:                // Simple byte-by-byte copy routine
  ldrb r3, [r1], #1    // Load from source, post-increment
  strb r3, [r0], #1    // Store to destination, post-increment
  sub r2, #1           // Decrement counter
  cmp r2, #0           // Check if finished
  bne memcpy           // Loop if more bytes remain
  bx lr                // Return from function
'''

initial_state = {
    'registers': {
        'r0': INPUT_STRING_BASE_ADDR,
        'r1': PASSWORD_BASE_ADDR,
        'r2': 0x0,
        'r3': 0x0,
        'r4': 0x0,
        'lr': RETURN_ADDRESS
    },
    'memory': {
        INPUT_STRING_BASE_ADDR: b'\x00\x00\x00\x00',
        INPUT_STRING_BASE_ADDR + 4: b'\x00\x00\x00\x00',
        PASSWORD_BASE_ADDR: b'SECR',
        PASSWORD_BASE_ADDR + 4: b'ET!\x00'
    }
}

success_state = {
    'registers': {
        'r0': 0x1
    }
}

challenge_4 = {
    'index': 4,
    'c_code': c_code,
    'asm_code': asm_code,
    'initial_state': initial_state,
    'success_state': success_state
}

print_challenge(challenge_4)

In [ ]:
run_challenge(challenge_4, debug=False)

# Challenge 5: Overflowing With Joy
The code copies the user supplied name into a name-buffer for later use by the progam. Since there was no length check you could supply a longer name than the 8 bytes that the programmer intended. However, the code is completely safe as the code only copies exactly 8 bytes and can therefore never overflow the destination buffer.

**Goal:** Can you overwrite the admin flag that is located just after the name buffer in the memory?

In [ ]:
# Challenge 5
# Run this block to pretty-print!

c_code = '''\
void copy_name(char *name_buffer, char* user_input) {
    memcpy(name_buffer, user_input, 8);
}

void memcpy(char *dest, char *src, int len) {
    for (int i=0; i<len; i++) {
        dest[i] = src[i];
    }
}
'''

NAME_STRING_BASE_ADDR = RAM_ADDRESS + 0x10
ADMIN_FLAG = RAM_ADDRESS + 0x18
INPUT_STRING_BASE_ADDR = RAM_ADDRESS + 0x100

asm_code = f'''\
push {{lr}}              // Save the return address (Link Register) onto the stack
mov r2, #8             // Set the length argument of memcpy to 8 bytes
bl memcpy              // Call the memcpy function (r0=dest, r1=src, r2=8)
pop {{pc}}               // Pop the saved lr into pc to return

memcpy:                // Simple byte-by-byte copy routine
  ldrb r3, [r1], #1    // Load from source, post-increment
  strb r3, [r0], #1    // Store to destination, post-increment
  sub r2, #1           // Decrement counter
  cmp r2, #0           // Check if finished
  bne memcpy           // Loop if more bytes remain
  bx lr                // Return from function
'''

initial_state = {
    'registers': {
        'r0': NAME_STRING_BASE_ADDR,
        'r1': INPUT_STRING_BASE_ADDR,
        'r2': 0x0,
        'r3': 0x0,
        'lr': RETURN_ADDRESS
    },
    'memory': {
        NAME_STRING_BASE_ADDR: b'\x00\x00\x00\x00',
        NAME_STRING_BASE_ADDR + 4: b'\x00\x00\x00\x00',
        ADMIN_FLAG: b'\x00',
        INPUT_STRING_BASE_ADDR: b'AAAA',
        INPUT_STRING_BASE_ADDR + 4: b'AAAA',
        INPUT_STRING_BASE_ADDR + 8: b'\x01\x00\x00\x00'
    }
}

success_state = {
    'memory': {
        ADMIN_FLAG: b'\x01'
    }
}

challenge_5 = {
    'index': 5,
    'c_code': c_code,
    'asm_code': asm_code,
    'initial_state': initial_state,
    'success_state': success_state
}

print_challenge(challenge_5)

In [ ]:
run_challenge(challenge_5, debug=False)

# Challenge 6: The Stack Balance Act
In response to the vulnerability found in the previous one, this challenge implements a length check of the string to make the system secure. You have entered 8 bytes of input data that is copied into a local buffer on the stack. The length of the data, assuming it is a string, is counted and returned. You have identified a new vulnerability and entered the string "AAAA" followed by four bytes that form an address to an unlock-function.

**Goal:** Can you gain control of the execution and jump to the unlock-function?

**Hint:** You will need to visualize the stack layout to understand this one.

In [ ]:
# Challenge 6
# Run this block to pretty-print!

USER_INPUT_BASE_ADDR = RAM_ADDRESS + 0x10

# The address of the function to unlock the system
UNLOCK_FUNCTION_ADDR = 0x08004444

c_code = '''\
int check_length(char *input) {
    char buffer[8];

    memcpy(buffer, input, 8);
    int len = strlen(buffer);

    return len;
}
'''

asm_code = f'''\
push {{r3, lr}}         // Save r3 and the return address (lr) to the stack (4 bytes each)
sub sp, #8            // Allocate an 8 byte buffer on the stack
mov r1, r0            // Set user input as source
mov r0, sp            // Set the newly allocated stack buffer as destination
mov r2, #8            // Set copy length to 8 bytes
bl memcpy             // Call the memcpy function (r0=dest, r1=src, r2=8)
mov r0, sp            // Reset r0 to point to the stack buffer
bl strlen             // Call the strlen function (r0=str). Length is returned in r0
add sp, #8            // Deallocate the 8 bytes of stack space
pop {{r3, pc}}          // Restore r3 and pop saved lr into pc to return to caller

memcpy:               // Simple byte-by-byte copy routine
  ldrb r3, [r1], #1   // Load from source, post-increment
  strb r3, [r0], #1   // Store to destination, post-increment
  sub r2, #1          // Decrement counter
  cmp r2, #0          // Check if finished
  bne memcpy          // Loop if more bytes remain
  bx lr               // Return from function

strlen:               // Counts the length of a string
  mov r1, #0          // Initialize counter to zero
loop:
  ldrb r2, [r0], #1   // Load one byte from input string, post-increment input pointer
  cmp r2, #0          // Check for null byte (end of the string)
  beq done            // If null byte found, exit loop
  add r1, r1, #1      // Otherwise, increment counter
  b loop              // Repeat
done:
  mov r0, r1          // Return counter in r0
  bx lr
'''

initial_state = {
    'registers': {
        'r0': USER_INPUT_BASE_ADDR,
        'r1': 0x0,
        'r2': 0x0,
        'r3': 0x44332211, # Some previous value that is being saved on the stack (pushed/popped) while the function runs
        'lr': RETURN_ADDRESS
    },
    'memory': {
        USER_INPUT_BASE_ADDR: b'AAAA',
        USER_INPUT_BASE_ADDR + 4: UNLOCK_FUNCTION_ADDR.to_bytes(4, 'little')
    }
}

success_state = {
    'registers': {
        'pc': UNLOCK_FUNCTION_ADDR
    }
}

challenge_6 = {
    'index': 6,
    'c_code': c_code,
    'asm_code': asm_code,
    'initial_state': initial_state,
    'success_state': success_state
}

print_challenge(challenge_6)

In [ ]:
run_challenge(challenge_6, debug=False)

# Your own challenge
This is where you **may** implement your own challenge idea. It does not have to be complex. If you don't want to implement it, **it is sufficient to describe it and its vulnerability in the report**.

In [ ]:
# Your own challenge
# Run this block to pretty-print!

USER_INPUT_BASE_ADDR = RAM_ADDRESS + 0x10

c_code = '''
YOUR CODE HERE
'''

asm_code = f'''\
YOUR CODE HERE
'''

initial_state = {
    'registers': {
        'r0': 0x0,
        'lr': RETURN_ADDRESS
    },
    'memory': {
        USER_INPUT_BASE_ADDR: b'AAAA',
    }
}

success_state = {
    'registers': {
        'r0': 0x1
    }
}

challenge_7 = {
    'index': 7,
    'c_code': c_code,
    'asm_code': asm_code,
    'initial_state': initial_state,
    'success_state': success_state
}

print_challenge(challenge_7)

In [ ]:
run_challenge(challenge_7, debug=False)